# Part A: GAN Improvements and Experiments

One sentence stating the goal: starting from the trained baseline DCGAN, isolate which changes help, then combine the winners into a single final model.

## 1. Imports and Setup

In [ ]:
# Import tensorflow/keras, numpy, matplotlib, json; set random seed for reproducibility.

In [ ]:
# Re-define the generator/discriminator/train_step builders from gan_baseline.ipynb directly here (not shared via a .py file), keeping this notebook self-contained for grading.

## 2. Load Baseline Results

One sentence noting we're loading the baseline's saved config, losses, and eye-test scores (no retraining) so every experiment has a fixed point of comparison.

In [ ]:
# Load CIFAR10 and apply the same preprocessing (normalize to [-1, 1], one-hot, val split) as the baseline notebook.

In [ ]:
# Load the baseline's saved JSON (config, final losses, eye-test scores) for reference throughout this notebook.

## 3. Comparison Metrics

One sentence stating the goal: define, before running any experiment, exactly how "better" will be measured and how the best setup will be picked, so results aren't judged after the fact by eyeballing.

### 3.1 Quality Score (primary metric)

One sentence defining the quality score: convert each image's eye-test label to a number (clear=1, marginal=0.5, nonsense=0) and average across the scored sample, per class and overall — the identical definition used in `vae_improvement.ipynb`, so GAN and VAE results stay comparable.

In [ ]:
# Define score_from_labels(): map a list of clear/marginal/nonsense tallies to a 0-1 quality score, reused by every experiment.

### 3.2 Discriminator/Generator Loss Balance (secondary / diagnostic metric)

One sentence defining the secondary metric: the final gap between generator and discriminator loss, tracked per model as a cheap automatic check for collapse or one network overpowering the other — the GAN analogue of the VAE's validation loss, since GANs have no single validation loss to rely on.

### 3.3 Decision Rule

One sentence stating the rule used to pick a winner: quality score is the primary ranking metric since it's what the assignment actually grades (image quality); the G/D loss balance is checked only as a tiebreaker or red flag (e.g. reject a high-quality-score model if training clearly collapsed) — this rule is fixed here, before any results exist, so it can't be quietly bent to favor a preferred outcome.

In [ ]:
# Initialize a shared results dict (starting with the baseline's numbers) that every experiment appends its quality score and loss balance to.
# Note: experiments 1-6 log metrics only and do NOT save .h5 weights — only the baseline and the Section 10 final model get saved weights.

## 4. Experiment 1: TTUR (Two Time-Scale Update Rule)

Hypothesis: the baseline uses one shared learning rate for both networks, which risks the discriminator learning faster than the generator and overpowering it; giving the discriminator a lower learning rate than the generator should raise the quality score by keeping their contest balanced for longer — checked against the risk of slowing convergence too much if the ratio is too extreme.

In [ ]:
# Build and train a GAN identical to baseline except for separate, unequal generator/discriminator learning rates (TTUR).

In [ ]:
# Generate a sample of images, score with score_from_labels() (Section 3.1), and log quality score + final G/D loss gap into the results dict.

In [ ]:
# Plot this experiment's generator/discriminator loss curves against the baseline's, on the same axes.

## 5. Experiment 2: Label Smoothing

Hypothesis: the baseline's hard 0/1 discriminator targets can make it overconfident and easy to saturate, so one-sided label smoothing (real target 0.9 instead of 1.0) should raise the quality score by keeping discriminator gradients informative for longer — checked against the risk of smoothing too much and weakening the discriminator's signal to the generator.

In [ ]:
# Build and train a GAN identical to baseline except the discriminator's real-image target is smoothed from 1.0 to 0.9.

In [ ]:
# Generate a sample of images, score with score_from_labels() (Section 3.1), and log quality score + final G/D loss gap into the results dict.

In [ ]:
# Plot this experiment's generator/discriminator loss curves against the baseline's, on the same axes.

## 6. Experiment 3: Spectral Normalization

Hypothesis: the baseline discriminator's weights are unconstrained and can grow to dominate the generator, so applying spectral normalization to the discriminator's conv layers should raise the quality score by capping its Lipschitz constant and stabilizing training — checked against the risk of over-constraining the discriminator so it can no longer distinguish real from fake effectively.

In [ ]:
# Build and train a GAN identical to baseline except the discriminator's conv layers are wrapped with spectral normalization.

In [ ]:
# Generate a sample of images, score with score_from_labels() (Section 3.1), and log quality score + final G/D loss gap into the results dict.

In [ ]:
# Plot this experiment's generator/discriminator loss curves against the baseline's, on the same axes.

## 7. Experiment 4: Color vs. Grayscale

Hypothesis (linked to the assignment's discussion question): converting inputs to grayscale removes color as a distinguishing cue, so the quality score should drop for classes that rely heavily on color but may hold up or even simplify shape-dominant classes — this experiment answers the question empirically rather than by reasoning alone.

In [ ]:
# Convert the training images to grayscale (single channel) and adjust the generator/discriminator's channel counts accordingly.

In [ ]:
# Build and train a GAN identical to baseline except for the single-channel input/output.

In [ ]:
# Generate a sample of images, score with score_from_labels() (Section 3.1) split by class, and log quality score + loss balance into the results dict.

In [ ]:
# Plot this experiment's generator/discriminator loss curves against the color baseline's, on the same axes.

### Discussion: Color vs. Grayscale, Empirically

One sentence stating whether this experiment's quality score confirmed or contradicted the prediction made in `gan_baseline.ipynb` Section 10.4, with the per-class score breakdown as evidence.

## 8. Experiment 5: Engineered Conditioning Features

Hypothesis: a bare one-hot label only tells the generator/discriminator "which of 10 buckets," so concatenating each class's mean per-channel color statistics (from `vae_eda.ipynb` Section 5, the same shared EDA both architectures use) alongside the one-hot vector should raise the quality score by giving both networks a richer conditioning signal — checked against the risk that the auxiliary features are redundant with one-hot or add noise the discriminator has to learn to ignore.

In [ ]:
# Compute each class's mean per-channel (R, G, B) statistics from the training set, building a small per-class auxiliary feature vector.

In [ ]:
# Build and train a GAN identical to baseline except the conditioning input is [one-hot label, auxiliary color-stats vector] instead of one-hot alone.

In [ ]:
# Generate a sample of images, score with score_from_labels() (Section 3.1), and log quality score + final G/D loss gap into the results dict.

In [ ]:
# Plot this experiment's generator/discriminator loss curves against the baseline's, on the same axes.

## 9. Experiment 6: Data Augmentation

Hypothesis: augmentation (e.g. random horizontal flip) exposes the discriminator to more visual variation per class without new data, which should raise the quality score by making the discriminator generalize instead of memorizing exact training images (a stronger discriminator gives the generator a better training signal) — checked against the risk that some flips are semantically wrong for a class and could confuse conditioning rather than help it.

In [ ]:
# Define an augmentation pipeline (e.g. random horizontal flip) applied to the training set only.

In [ ]:
# Build and train a GAN identical to baseline except the discriminator's real-image inputs are passed through the augmentation pipeline.

In [ ]:
# Generate a sample of images, score with score_from_labels() (Section 3.1), and log quality score + final G/D loss gap into the results dict.

In [ ]:
# Plot this experiment's generator/discriminator loss curves against the baseline's, on the same axes.

## 10. Final Model

One sentence stating the goal: apply the decision rule from Section 3.3 to combine the winning settings from experiments 1-3, 5, and 6 (TTUR, label smoothing, spectral normalization, engineered conditioning, augmentation) into the single final model — Experiment 4 (color vs. grayscale) targets a discussion question, not a quality lever, so it's evaluated separately and not folded into the combination. This is the only model besides the baseline whose weights get saved.

### 10.1 Select Best Settings

In [ ]:
# Read each experiment's quality score/loss balance from the results dict, and pick the best setting per axis using the Section 3.3 decision rule.

### 10.2 Train the Final Model

In [ ]:
# Build and train a GAN using the combined best settings from 10.1 (no weights saved yet — training only).

In [ ]:
# Save the final model's generator weights to .h5 — the deliverable weights file for this notebook, alongside the baseline's.

### 10.3 Evaluate the Final Model

In [ ]:
# Generate 1000 images (100/class) with the final model, score with score_from_labels(), and log into the results dict against the baseline.

In [ ]:
# Plot the final model's generator/discriminator loss curves against the baseline's, on the same axes.

## 11. Ablation Summary

One sentence noting this table pulls the shared results dict into one place, so the final model's choice is backed by numbers already computed during the experiments, not a new comparison step.

In [ ]:
# Build one ablation table from the results dict: baseline, each experiment, and the final model, with quality score and G/D loss balance as columns.

In [ ]:
# Bar chart comparing quality scores across all models.

## 12. Conclusion

One sentence summarizing which single changes raised the quality score and which didn't (per the Section 3 metrics), how much the final model gained over the baseline, and noting its weights are saved to .h5 alongside the baseline's as the Part A GAN deliverable, feeding into `vae_gan_comparison.ipynb`.